In [18]:
import json
import requests
from lxml import html
import pandas as pd

# configurations

In [4]:
# read configurations (for any changes, update the config.json file and execute cells again)

with open("config.json", 'r') as file:
    configs = json.load(file)

configs

{'input_filepath': 'data/input_urls.json',
 'output_filepath': 'scraped_data.tsv',
 'required_info': [{'column_name': 'text',
   'xpath': "//div[@class='govspeak']//text()"},
  {'column_name': 'date',
   'xpath': '/html/body/div[2]/main/div[2]/div/div[1]/div/dl/dd[2]//text()',
   'index': 0},
  {'column_name': 'profile_link',
   'xpath': '/html/body/div[2]/main/div[2]/div/div[1]/div/dl/dd[1]/a[2]/@href',
   'index': 0}]}

# Read input

In [6]:
# input file as dictionary with domain as key and list of urls as values.

with open(configs["input_filepath"], 'r') as file:
    input_urls = json.load(file)

input_urls

{'https://example.com': ['/'],
 'https://www.gov.uk': ['government/speeches/chancellor-john-healeys-growth-speech-2026',
  'government/speeches/autumn-budget-2024-speech',
  'government/speeches/spring-budget-2024-speech',
  'government/speeches/spring-budget-2023-speech',
  'government/speeches/the-autumn-statement-2022-speech',
  'government/speeches/budget-speech-2021',
  'government/speeches/budget-speech-2020',
  'government/speeches/spring-statement-2019-philip-hammonds-speech']}

# Apply filters

In [24]:
#format output dictionary
output = {}
for domain, urls in input_urls.items():
    
    #iterate through all urls for a given domain
    for url in urls:
        speech_page = requests.get(domain + '/' + url)
        speech_soup = html.fromstring(speech_page.content)
        
        # format output
        if 'domain' not in output.keys():
            output['domain'] = []
        output['domain'].append(domain)

        if 'url' not in output.keys():
            output['url'] = []
        output['url'].append(url)

        # apply filters
        for info in configs["required_info"]:
            values = []
            if 'xpath' in info.keys():
                value = speech_soup.xpath(info['xpath'])
            if ('index' in info.keys()) & (len(values) > 0):
                value = value[int(info['index'])]
            if info['column_name'] not in output.keys():
                output[info['column_name']] = []
            output[info['column_name']].append(value)
            

# Write output

In [25]:
df_output = pd.DataFrame(output)
df_output.to_csv('data/scraped_data.tsv', sep='\t', index=False)